# Projet intégrateur — QML complet

**Semaine 14 — Cours de Quantum Machine Learning (Master/PhD)**

## Objectifs

Ce notebook est un **template prêt à remplir** pour réaliser un projet complet de Quantum Machine Learning, du choix du problème à l'analyse des résultats, en passant par l'implémentation et le benchmark.

### Projets au choix :
1. **Kernel quantique à 40+ qubits avec BFT** : Implémentez un fidelity kernel + Bit-Flip Tolerance sur IBM Quantum
2. **Transfer learning quantique** : CNN (ResNet18) gelé + tête VQC sur Hymenoptera (torchvision)
3. **QAOA pour optimisation financière** : MaxCut sur problème de portefeuille
4. **Benchmark systématique QML vs. classique** : 5 datasets, 3 modèles quantiques, 3 classifieurs classiques

**Bibliothèques** : PennyLane, Qiskit ML, PyTorch, scikit-learn, matplotlib

---
## 0. Configuration et imports

In [ ]:
# ── Imports généraux ──
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.datasets import load_iris, load_wine, make_classification
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 5)})
print("Configuration initiale terminée.")

---
## 1. Choix du projet

Sélectionnez l'un des projets ci-dessous en décommentant la ligne correspondante.

In [ ]:
# ── Choix du projet ──
PROJECT = 1  # 1: Kernel BFT, 2: Transfer learning, 3: QAOA, 4: Benchmark

project_names = {
    1: "Kernel quantique à 40+ qubits avec BFT",
    2: "Transfer learning quantique (CNN + VQC)",
    3: "QAOA pour optimisation combinatoire",
    4: "Benchmark systématique QML vs. classique",
}

print(f"Projet choisi : {project_names[PROJECT]}")

---
## 2. Implémentation

### 2.1 Chargement des données

Les jeux de données disponibles : Iris, Wine, synthétiques, ou Hymenoptera (torchvision).

In [ ]:
def load_dataset(name="iris"):
    if name == "iris":
        data = load_iris()
        X, y = data.data, data.target
        # Binarisation pour classification binaire (sinon 3 classes)
        # y = (y == 0).astype(int)  # décommenter pour binaire
    elif name == "wine":
        data = load_wine()
        X, y = data.data, data.target
        y = (y == 0).astype(int)  # binarisation
    elif name == "synthetic":
        X, y = make_classification(n_samples=300, n_features=6, n_informative=4,
                                    n_redundant=2, n_classes=2, random_state=42)
    else:
        raise ValueError(f"Dataset '{name}' inconnu.")
    
    X = StandardScaler().fit_transform(X)
    return train_test_split(X, y, test_size=0.3, random_state=42)

# Chargement
X_train, X_test, y_train, y_test = load_dataset("iris")
print(f"Train : {X_train.shape}, Test : {X_test.shape}")
print(f"Features : {X_train.shape[1]}, Classes : {len(np.unique(y_train))}")

### 2.2 Modèle quantique

In [ ]:
# ── Choix du framework et du modèle ──
FRAMEWORK = "pennylane"  # "pennylane", "qiskit", "cirq"

if FRAMEWORK == "pennylane":
    import pennylane as qml
    from pennylane import numpy as np_pl
    
    n_qubits = 4
    n_layers = 2
    dev = qml.device("default.qubit", wires=n_qubits)
    
    @qml.qnode(dev, diff_method="parameter-shift")
    def vqc(params, x):
        qml.AngleEmbedding(x[:n_qubits], wires=range(n_qubits))
        for l in range(n_layers):
            for i in range(n_qubits):
                qml.RX(params[l, i, 0], wires=i)
                qml.RY(params[l, i, 1], wires=i)
            for i in range(n_qubits - 1):
                qml.CNOT(wires=[i, i + 1])
        return qml.expval(qml.PauliZ(0))
    
    print("Modèle PennyLane VQC défini.")

### 2.3 Entraînement

In [ ]:
def train_vqc(X_tr, y_tr, X_te, y_te, n_epochs=100):
    np.random.seed(42)
    params = np_pl.random.uniform(-np.pi, np.pi, size=(n_layers, n_qubits, 2))
    opt = qml.AdamOptimizer(stepsize=0.1)
    
    train_losses = []
    test_accs = []
    
    for epoch in range(n_epochs):
        loss = 0.0
        for x, y in zip(X_tr, y_tr):
            pred = vqc(params, x)
            loss += (pred - y) ** 2
        loss /= len(X_tr)
        train_losses.append(loss)
        
        # Mise à jour des paramètres
        def cost_fn(p):
            total = 0.0
            for x, y in zip(X_tr, y_tr):
                total += (vqc(p, x) - y) ** 2
            return total / len(X_tr)
        
        params = opt.step(cost_fn, params)
        
        # Accuracy test
        preds = []
        for x in X_te:
            preds.append(1.0 if vqc(params, x) >= 0.0 else 0.0)
        acc = accuracy_score(y_te, preds)
        test_accs.append(acc)
        
        if (epoch + 1) % 20 == 0:
            print(f"Étape {epoch+1:3d} : loss = {loss:.4f}, test acc = {acc:.4f}")
    
    return params, train_losses, test_accs

print("Entraînement du VQC...")
params_trained, losses, accs = train_vqc(X_train, y_train, X_test, y_test, n_epochs=80)

### 2.4 Baseline classique

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

classifiers = {
    "SVM (RBF)": SVC(kernel="rbf", C=1.0),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42),
}

baseline_results = {}
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average="weighted")
    baseline_results[name] = {"accuracy": acc, "f1_score": f1}
    print(f"{name:20s} : accuracy = {acc:.4f}, F1 = {f1:.4f}")

---
## 3. Résultats et analyse

In [ ]:
# ── Comparaison des performances ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Courbe d'apprentissage
axes[0].plot(losses, label="Loss (train)", lw=2)
axes[0].set_xlabel("Époque")
axes[0].set_ylabel("MSE")
axes[0].set_title("Convergence du VQC")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Barplot comparatif
all_models = {"VQC (quantique)": accuracy_score(y_test, [1.0 if vqc(params_trained, x) >= 0 else 0.0 for x in X_test])}
for name, res in baseline_results.items():
    all_models[name] = res["accuracy"]

bars = axes[1].barh(list(all_models.keys()), list(all_models.values()), color=["blue", "gray", "gray", "gray"])
axes[1].set_xlabel("Accuracy (test)")
axes[1].set_title("Comparaison quantique vs. classique")
axes[1].grid(True, axis="x", alpha=0.3)
for bar, val in zip(bars, all_models.values()):
    axes[1].text(val + 0.01, bar.get_y() + bar.get_height() / 2, f"{val:.3f}",
                va="center", fontsize=10)

plt.tight_layout()
plt.show()

print("\n=== Analyse des résultats ===")
for name, res in baseline_results.items():
    print(f"{name:20s} : acc = {res['accuracy']:.4f}")
print(f"{'VQC (quantique)':20s} : acc = {list(all_models.values())[0]:.4f}")

### Analyse approfondie

**Questions à explorer dans votre rapport :**
1. Le modèle quantique surpasse-t-il la baseline classique ? Si oui, pourquoi ?
2. Quelles sont les limitations (nombre de qubits, profondeur du circuit, barren plateaux) ?
3. Quelle stratégie d'encodage a été utilisée et pourquoi ?
4. Quel est l'impact du bruit si l'on passe sur du matériel réel ?
5. Le projet est-il scalable à plus grande échelle ?

In [ ]:
# ── Métriques détaillées ──
y_pred_vqc = []
for x in X_test:
    y_pred_vqc.append(1.0 if vqc(params_trained, x) >= 0.0 else 0.0)

print("Rapport de classification VQC :")
print(classification_report(y_test, y_pred_vqc))

print("\nMatrice de confusion VQC :")
print(confusion_matrix(y_test, y_pred_vqc))

---
## 4. Conclusion

Rédigez ici une synthèse de votre projet :
- Résumé des objectifs
- Principaux résultats (inclure des chiffres)
- Limites identifiées
- Perspectives et améliorations possibles
- Leçons apprises

---
## Annexes : squelette pour les autres projets

Décommentez la section correspondant à votre choix.

### Projet 1 : Kernel quantique avec BFT

```python
# from qiskit_machine_learning.kernels import FidelityQuantumKernel
# from qiskit.circuit.library import ZZFeatureMap
# 
# feature_map = ZZFeatureMap(feature_dimension=4, reps=2)
# kernel = FidelityQuantumKernel(feature_map=feature_map)
# kernel_matrix_train = kernel.evaluate(x_vec=X_train)
# # Implémenter BFT : seuil de poids de Hamming
# # Voir [Agl26] pour les détails
```

### Projet 2 : Transfer learning quantique

```python
# import torch
# import torchvision.models as models
# from torchvision.datasets import ImageFolder
# 
# # Backbone ResNet18 gelé
# backbone = models.resnet18(pretrained=True)
# for param in backbone.parameters():
#     param.requires_grad = False
# backbone.fc = torch.nn.Identity()  # supprimer la dernière couche
# # Sortie : 512 features -> VQC 8 qubits
# # Voir [Tra25] pour les détails
```

### Projet 3 : QAOA pour optimisation financière

```python
# from qiskit_optimization.applications import PortfolioOptimization
# from qiskit_algorithms import QAOA
# from qiskit_algorithms.optimizers import COBYLA
# from qiskit_optimization.algorithms import MinimumEigenOptimizer
# 
# # Problème de portefeuille : n actifs, rendement attendu, covariance
# portfolio = PortfolioOptimization(
#     expected_returns=np.array([0.1, 0.15, 0.12, 0.08, 0.11]),
#     covariances=np.eye(5) * 0.05,
#     risk_factor=0.5,
#     budget=2
# )
```

### Projet 4 : Benchmark systématique QML vs. classique

```python
# datasets = ["iris", "wine", "breast_cancer", "digits", "synthetic"]
# models_quantum = ["VQC (PennyLane)", "QSVM (Qiskit)"]
# models_classical = ["SVM", "Random Forest", "MLP", "XGBoost"]
# # Pour chaque dataset : entraîner tous les modèles, collecter acc, F1, temps
# # Afficher un heatmap des performances
```

---
## Exercices — Pour aller plus loin

1. **Analyse de la variance** : Relancez l'entraînement 10 fois avec des seeds différentes et tracez la moyenne +/- écart-type.
2. **Optimisation d'hyperparamètres** : Grid search sur le nombre de couches, le learning rate, le nombre de qubits.
3. **Étude de scalabilité** : Augmentez progressivement la dimensionnalité des données (2, 4, 6, 8 features) et mesurez l'accuracy.
4. **Interprétabilité** : Utilisez `qml.draw` ou `qml.draw_mpl` pour visualiser le circuit entraîné.
5. **Simulateur avec bruit** : Activez un bruit réaliste (dépolarisant, amplitude damping) et étudiez sa dégradation.
6. **Comparaison multi-framework** : Implémentez le même VQC en Qiskit et Cirq, comparez les performances.